In [ ]:
from google.colab import userdata
key = userdata.get('HUGGINGFACEHUB_API_TOKEN')

In [ ]:
! pip install grandalf
!pip install --upgrade langchain_huggingface
!pip install langchain_community
!pip install langchain_openai

## chatmodels
- using hugging face
- using openai

In [ ]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text2text-generation",
    huggingfacehub_api_token=key,
)
model = ChatHuggingFace(llm=llm)

# Removed ChatHuggingFace wrapper as flan-t5-small is a text-to-text model
response = model.invoke("What is the capital of Pakistan?")
print(response.content)

In [ ]:
# from langchain.chat_models import ChatOpenAI

from langchain_openai import ChatOpenAI


In [ ]:
# notes: now I don't have paid chatgpt so after late you can try
# model = ChatOpenAI(model='gpt-4', temperature=1.5, max_completion_tokens=10)

# result = model.invoke("Write a 5 line poem on cricket")

# print(result.content)

# embeddings
- query embedding
- document embedding
- similarity -->both cover query and document

- query embedding

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings


In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

vector = embedding.embed_query('Islambad is the capital of Pakistan')

vector


In [ ]:
print(len(vector))

- doc embedding

In [ ]:
doc_embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

documents = [
    "Delhi is the capital of India",
    "Kolkata is the capital of West Bengal",
    "Paris is the capital of France"
]


reuslt = doc_embedding.embed_documents(documents)
print(reuslt)

- similirity

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)


documents = [
    "Virat Kohli is an Indian cricketer known for his aggressive batting and leadership.",
    "MS Dhoni is a former Indian captain famous for his calm demeanor and finishing skills.",
    "Sachin Tendulkar, also known as the 'God of Cricket', holds many batting records.",
    "Rohit Sharma is known for his elegant batting and record-breaking double centuries.",
    "Jasprit Bumrah is an Indian fast bowler known for his unorthodox action and yorkers."
]

query = 'tell me about bumrah'

document_embeddings = embedding.embed_documents(documents)
query_embedding = embedding.embed_query(query)

scores = cosine_similarity([query_embedding], document_embeddings)
index, score = sorted(list(enumerate(scores)),key=lambda x:x[1])[-1]

print(query)
print(documents[index])
print("similarity score is:", score)

# prompot

In [ ]:
!pip install --upgrade langchain langchain-huggingface --quiet
!pip install --upgrade sentence-transformers --quiet


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# HuggingFace model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task='text-generation',
    huggingfacehub_api_token=key,
)

model = ChatHuggingFace(llm=llm)

# Prompt template
template1 = PromptTemplate(
    input_variables=["product"],
    template="What is a good name for a company that makes {product}?",
)

# Invoke prompt
prompt = template1.invoke({"product": "colorful socks"})
result = model.invoke(prompt)

print(result.content)


# messages
- SystemMessage
-  HumanMessage
-  AIMessage

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct", # Changed to a known working instruct model
    task = "text-generation",
    huggingfacehub_api_token=key
)
model = ChatHuggingFace(llm=llm)

messages = [
    SystemMessage(content="You are a helpful assistant "),
    HumanMessage(content="tell me about langchain"),
]
result = model.invoke(messages)
messages.append(AIMessage(content=result.content))
print(messages)

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate


llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct", # Changed to a known working instruct model
    task = "text-generation",
    huggingfacehub_api_token=key
)
model = ChatHuggingFace(llm=llm)



# template
template = PromptTemplate(
    template="""
Please summarize the research paper titled "{paper_input}" with the following specifications:
Explanation Style: {style_input}
Explanation Length: {length_input}
1. Mathematical Details:
   - Include relevant mathematical equations if present in the paper.
   - Explain the mathematical concepts using simple, intuitive code snippets where applicable.
2. Analogies:
   - Use relatable analogies to simplify complex ideas.
If certain information is not available in the paper, respond with: "Insufficient information available" instead of guessing.
Ensure the summary is clear, accurate, and aligned with the provided style and length.
""",
input_variables=['paper_input', 'style_input','length_input'],
validate_template=True
)

template.save('template.json')


chat_history = [
    SystemMessage(content='You are a helpful AI assistant')
]

while True:
    user_input = input('You: ')
    chat_history.append(HumanMessage(content=user_input))
    if user_input == 'exit':
        break
    result = model.invoke(chat_history)
    chat_history.append(AIMessage(content=result.content))
    print("AI: ",result.content)

print(chat_history)

# output-parsers

- StrOutputParser
- pydanticoutputparser
- stroutputparser

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser,PydanticOutputParser,StrOutputParser

- StrOutputParser

In [ ]:

# Define the model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)

model = ChatHuggingFace(llm=llm)


parse = StrOutputParser()

# 1st prompt -> detailed report
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

# 2nd prompt -> summary
template2 = PromptTemplate(
    template='Write a 5 line summary on the following text. /n {text}',
    input_variables=['text']
)

# chain
chain = template1 | model | parse | template2 | model | parse

chain.invoke({'topic':'Cricket'})

- PydanticOutputParser

In [ ]:
from pydantic import Field,BaseModel

# Define the model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)

model = ChatHuggingFace(llm=llm)

class Person(BaseModel):
    name: str = Field(description='name of the person')
    age: int = Field(gt=18,description='age of the person')
    city: str = Field(description='city of the person')

parser = PydanticOutputParser(pydantic_object=Person)

template = PromptTemplate(
    template='Generate the name, age and city of a fictional {place} person \n {format_instruction}',
    input_variables=['place'],
    partial_variables={'format_instruction':parser.get_format_instructions()}
)

chain = template | model | parser

final_result = chain.invoke({'place':'Pakistan'})

print(final_result)

- JsonOutputParser

In [ ]:
# Define the model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)

model = ChatHuggingFace(llm=llm)

parser = JsonOutputParser()

template = PromptTemplate(
    template='Give me 5 facts about {topic} \n {format_instruction}',
    input_variables=['topic'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

chain = template | model | parser

result = chain.invoke({'topic':'black hole'})

print(result)

# chains
- sequential_chain/simple chain
- paraller chain
- condition chain

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableBranch

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)

model = ChatHuggingFace(llm=llm)

prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)


parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({'topic':'cricket'})

print(result)

chain.get_graph().print_ascii()

In [ ]:
from langchain_core.runnables import RunnableParallel


llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)

model1 = ChatHuggingFace(llm=llm)

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)

model2 = ChatHuggingFace(llm=llm)



prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})

print(result)

chain.get_graph().print_ascii()

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Load models
llm1 = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)
model1 = ChatHuggingFace(llm=llm1)

llm2 = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)
model2 = ChatHuggingFace(llm=llm2)

# Prompts
prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n{text}',
    input_variables=['text']
)
prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n{text}',
    input_variables=['text']
)
prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

text = """Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection...
[rest of your text]
"""

# Function to run a model chain
def run_chain(prompt, model, text):
    result = model.invoke(prompt.invoke({'text': text}))
    return parser.parse(result.content)

# Run notes and quiz **in parallel**
with ThreadPoolExecutor() as executor:
    futures = {
        'notes': executor.submit(run_chain, prompt1, model1, text),
        'quiz': executor.submit(run_chain, prompt2, model2, text)
    }
    results = {k: f.result() for k, f in futures.items()}

# Merge results
merge_prompt = prompt3.invoke({'notes': results['notes'], 'quiz': results['quiz']})
final_result = model1.invoke(merge_prompt)
final_text = parser.parse(final_result.content)

print(final_text)


- conditon chain

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

llm1 = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)
model1 = ChatHuggingFace(llm=llm1)

llm2 = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)
model2 = ChatHuggingFace(llm=llm2)

parser = StrOutputParser()

class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))

chain.get_graph().print_ascii()

# document-loaders
- TextLoader
- PyPDFLoader
- DirectoryLoader

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader,PyPDFLoader,DirectoryLoader,CSVLoader

- TextLoader

In [ ]:
llm1 = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=key
)
model1 = ChatHuggingFace(llm=llm1)

prompt = PromptTemplate(
    template='Write a summary for the following poem - \n {cricket}',
    input_variables=['cricket']
)

parser = StrOutputParser()

loader = TextLoader('cricket.txt')
docs = loader.load()

print(type(docs))

print(len(docs))

print(docs[0].page_content)

print(docs[0].metadata)

chain = prompt | model1 | parser

print(chain.invoke({'cricket':docs[0].page_content}))

- PyPDFLoader

In [ ]:
! pip install pypdf

In [ ]:
loader = PyPDFLoader('attention.pdf')
docs = loader.load()
print(len(docs))

print(docs[0].page_content)
print(docs[1].metadata)

- DirectoryLoader

In [ ]:
loader = DirectoryLoader('books',glob='**/*.pdf', loader_cls=PyPDFLoader)
docs = loader.lazy_load()

for document in docs:
    print(document.metadata)

- csv loader

In [ ]:
loader = CSVLoader(file_path='train.csv')
docs = loader.load()

print(len(docs))
print(docs[1])

# text-splitters

- length_based spliter
- text_structure_based spliter
- markerdown spliter
- python code spliter

In [ ]:
# lenght based spliter
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader('attention.pdf')
docs = loader.load()

spliter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
    separator=''
)

result = spliter.split_documents(docs)
print(result[1].page_content)


In [ ]:
# text structure based spliter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
Space exploration has led to incredible scientific discoveries. From landing on the Moon to exploring Mars, humanity continues to push the boundaries of what’s possible beyond our planet.

These missions have not only expanded our knowledge of the universe but have also contributed to advancements in technology here on Earth. Satellite communications, GPS, and even certain medical imaging techniques trace their roots back to innovations driven by space programs.
"""

spliters = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
)

result = spliters.split_text(text)
print(result[1])


print(len(result))



In [ ]:
# markerdown text spliter
from langchain_text_splitters import MarkdownHeaderTextSplitter,Language
text = """
# Project Name: Smart Student Tracker

A simple Python-based project to manage and track student data, including their grades, age, and academic status.


## Features

- Add new students with relevant info
- View student details
- Check if a student is passing
- Easily extendable class-based design


## 🛠 Tech Stack

- Python 3.10+
- No external dependencies


## Getting Started

1. Clone the repo
   ```bash
   git clone https://github.com/your-username/student-tracker.git

"""

spliter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=200,
    chunk_overlap=0,
)

chunks = spliter.split_text(text)
print(len(chunks))
print(chunks[0])
#

In [ ]:
# python code spliter

from langchain_text_splitters import RecursiveCharacterTextSplitter,Language

text = """
class Student:
    def __init__(self, name, age, grade):
        self.name = name
        self.age = age
        self.grade = grade  # Grade is a float (like 8.5 or 9.2)

    def get_details(self):
        return self.name"

    def is_passing(self):
        return self.grade >= 6.0


# Example usage
student1 = Student("Aarav", 20, 8.2)
print(student1.get_details())

if student1.is_passing():
    print("The student is passing.")
else:
    print("The student is not passing.")

"""

# Initialize the splitter
splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=300,
    chunk_overlap=0,
)

# Perform the split
chunks = splitter.split_text(text)

print(len(chunks))
print(chunks[1])

# vector store

In [ ]:
! pip install chromadb tiktoken

In [ ]:
!pip install langchain-community sentence-transformers
! pip install chromadb

In [ ]:
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc chromadb
!pip install --upgrade opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc chromadb

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

doc1 = Document(
    page_content=(
        "Babar Azam is one of the most successful and consistent batsmen in PSL history. "
        "Known for his elegant batting style and reliability, he has led Karachi Kings in multiple seasons."
    ),
    metadata={"team": "Karachi Kings"}
)

doc2 = Document(
    page_content=(
        "Shaheen Shah Afridi is the most successful young captain in PSL history, "
        "leading Lahore Qalandars to consecutive titles. He is known for his calm temperament and "
        "ability to perform in high-pressure situations."
    ),
    metadata={"team": "Lahore Qalandars"}
)

doc3 = Document(
    page_content=(
        "Sarfaraz Ahmed, famously known for his composed leadership, has led Quetta Gladiators to a PSL title. "
        "His wicketkeeping, finishing abilities, and leadership are widely respected."
    ),
    metadata={"team": "Quetta Gladiators"}
)

doc4 = Document(
    page_content=(
        "Haris Rauf is considered one of the most explosive fast bowlers in T20 cricket. "
        "Playing for Lahore Qalandars, he is known for his pace, aggression, and death-over skills."
    ),
    metadata={"team": "Lahore Qalandars"}
)

doc5 = Document(
    page_content=(
        "Shadab Khan is a dynamic all-rounder who contributes with both bat and ball. "
        "Representing Islamabad United, his sharp fielding and match-winning performances make him a key player."
    ),
    metadata={"team": "Islamabad United"}
)

docs = [doc1, doc2, doc3, doc4, doc5]

vector_store = Chroma(
    collection_name='cricketers',
    embedding_function=HuggingFaceEmbeddings(),
    persist_directory='my_chroma_db',
)

In [ ]:
# add documents
vector_store.add_documents(docs)

In [ ]:
vector_store.get(include=['documents','embeddings','metadatas'])


In [ ]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

In [ ]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

In [ ]:
vector_store.similarity_search_with_score(
    query='',
    filter={"team": "Lahore Qalandars"},
    k=2
)

In [ ]:
# update documents
updated_doc1 = Document(
    page_content=(
        "Shaheen Shah Afridi, the dynamic captain of Lahore Qalandars, is celebrated for his aggressive leadership "
        "and lethal fast bowling. Under his captaincy, Lahore Qalandars won back-to-back PSL titles, marking one of the "
        "most successful eras in the league. Shaheen’s ability to strike early with the new ball and deliver crucial "
        "overs at the death has established him as one of T20 cricket’s most impactful bowlers. His leadership style, "
        "combined with his consistency and match-winning spells, has made him a defining figure in PSL history."
    ),
    metadata={"team": "Lahore Qalandars"}
)
vector_store.update_document(document_id='a721020b-c843-452c-8805-3eb24455d320',document=updated_doc1)


In [ ]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

In [ ]:
vector_store.delete(ids='a721020b-c843-452c-8805-3eb24455d320')

In [ ]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])


# retrievers
- Wikipedia Retriever
- Vector Store Retriever
- MMR
- Multiquery Retriever
- ContextualCompressionRetriever

In [ ]:
!pip install  faiss-cpu wikipedia

- Wikipedia Retriever

In [ ]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_community.utilities import WikipediaAPIWrapper
import wikipedia

# 1. Force the underlying library to use a descriptive User-Agent
wikipedia.set_user_agent("MyGeoPoliticsBot/1.0 (contact: myemail@example.com)")

# 2. Create the wrapper manually
api_wrapper = WikipediaAPIWrapper(top_k_results=2, lang='en')

# 3. Pass the wrapper into the retriever
retriever = WikipediaRetriever(wikipedia_api_wrapper=api_wrapper)

# 4. Run your query
query = "the geopolitical history of india and pakistan"
docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content[:200]}...")


- Vector Store Retriever

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document


# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

# Step 2: Initialize embedding model
embedding_model = HuggingFaceEmbeddings()

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

query = "What is Chroma used for?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

In [ ]:
results = vectorstore.similarity_search(query, k=2)
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

- MMR

MMR yeh ensure karta hai:

- Pehla result → most relevant
- Dusra result → relevant + different from first
- Teesra result → relevant + diverse

👉 Matlab:

- Ek LangChain ka intro
- Ek uska use-case
- Ek embeddings ya tools ka mention

In [ ]:
from langchain_community.vectorstores import FAISS
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]


# Initialize OpenAI embeddings
embedding_model = HuggingFaceEmbeddings()

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance 1 mean normal result and 0 mean randomness add and 0.5 mean half half both normal + randomness
)

query = "What is langchain?"
results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

- Multiquery Retriever

# Rag
- we will cover rag concept in rag module


# Tools

In [ ]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

### Built-in Tool
-  DuckDuckGo Search
- Shell Tool

-  DuckDuckGo Search

In [ ]:
!pip install ddgs

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

In [ ]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)
print(search_tool.args_schema.model_json_schema())

- Shell Tool

In [ ]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

Custom Tools
- i have three method
- tool
- structuretool
- basetool

- tool
- most use this for start

In [ ]:
from langchain_community.tools import tool

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
  "Multiply two number"
  return a* b


result = multiply.invoke({"a":3, "b":5})
print(result)

In [ ]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

In [ ]:
print(multiply.args_schema.model_json_schema())

- Method 2 - Using StructuredTool

In [ ]:
from langchain_community.tools import StructuredTool
from pydantic import BaseModel, Field


class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")


def multiply_func(a: int, b: int) -> int:
    return a * b


multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)


result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)


- Method 3 - Using BaseTool Class

In [ ]:
from langchain.tools import BaseTool
from typing import Type


# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")


class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b


multiply_tool = MultiplyTool()


result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

### Toolkit

In [ ]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [ ]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]


In [ ]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)
